# 04 データ量とノイズ — 深層学習はデータが多いほど有利か

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 10分 ＋ 実行 数秒（比較表は事前に `src/run_phase4.py`（約20分）を実行した結果のCSVを読みます）

## ゴール
- 学習期間を **1.5年・3年・5年・10年**と変え、手法の成績がどう変わるか（**学習曲線**）を見る。
- 「深層学習は大量のデータが必要」という通説が、この問題で成り立つかを確かめる。
- **実データ並みの短さ**（約1.5年）で、どの手法が頼りになるかを知る。
- 発電量の**ノイズ**を変えて、隠れた状態を読む価値が、ノイズでどう変わるかを見る。

## 前のNotebookとのつながり
03 で、GRU は予報が完璧な世界で隠れた状態を読めました。しかし学習期間は10年。実データ（`ml_forecast`）は約1.5年分しかありません。**短いデータでも、同じことが言えるでしょうか**。

## 実験の設計
- 世界A（普通）と世界B（予報が完璧）。各4世界、NN系は3つの初期値の平均。
- 学習期間が短いときは、早期終了のための内側の検証を、学習期間の1/4までにします（1.5年だと約4か月）。
- ⚠️ 検証期間（2年）は共通です。

## キーワード
- **学習曲線**: 学習データ量を横軸にした、成績のグラフ。
- **データ効率**: 少ないデータでどれだけ良い成績が出るか。
- **ノイズの床**: 予測できない揺らぎ（発電量の3%など）による、誤差の下限。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 220)

OUT = ROOT / "outputs"
if not (OUT / "phase4_E1.csv").exists():
    raise SystemExit("先に .venv/bin/python src/run_phase4.py を実行してください（約20分）")
e1, e2 = pd.read_csv(OUT / "phase4_E1.csv"), pd.read_csv(OUT / "phase4_E2.csv")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, w in zip(ax, ["A 普通", "B 予報が完璧"]):
    for m in ["予報の直線回帰", "HGB（全特徴量）", "HGB（予報・季節だけ）", "MLP（予報・季節だけ）", "GRU（生の窓）", "上限2"]:
        g = e1[(e1["世界"] == w) & (e1["手法"] == m)]
        a.errorbar(g["学習期間（年）"], g["MAE"], yerr=g["標準誤差"], fmt="o-", capsize=2, label=m, ls="--" if m == "上限2" else "-")
    a.set_xscale("log"); a.set_xticks([1.5, 3, 5, 10]); a.set_xticklabels([1.5, 3, 5, 10])
    a.set_xlabel("学習期間（年）"); a.set_ylabel("検証のMAE"); a.set_title(f"世界{w}"); a.grid(alpha=0.3)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

**出力の見方**（学習曲線。左が普通の世界A、右が予報が完璧な世界B）:
- **全体の傾向**: 直線回帰（学習が2パラメータだけ）は、学習期間によらず**一定**（A: 0.86、B: 0.30）。ほかの手法は、データが増えるほど**良くなります**。
- 世界Bでは、データが増えるほど、直線回帰との差が開きます。**隠れた状態を読むには、ある程度のデータが要る**。

In [ ]:
e1.pivot_table(index="手法", columns=["世界", "学習期間（年）"], values="MAE").loc[
    ["予報の直線回帰", "HGB（全特徴量）", "HGB（予報・季節だけ）", "MLP（予報・季節だけ）", "GRU（生の窓）", "上限2"]].round(3)

**出力の見方**（MAE。標準誤差は 0.006〜0.018）:

**世界A（普通）**:
- **1.5年（実データ並み）**: 直線回帰 0.867、HGB（全特徴量）**0.895**、HGB（予報・季節だけ）0.873、GRU 0.859、**MLP（予報・季節だけ）0.836**。**HGB（全特徴量）は、直線回帰より悪い**（過学習）。GRUも直線回帰とほぼ同じ（−0.008）。**データが少ないと、複雑な手法は単純な基準に勝てません**。MLP（予報・季節だけ）だけが、直線回帰を上回る（−0.031）。
- **10年**: MLP 0.761、GRU 0.769、HGB（全特徴量）0.786、直線回帰 0.861。**ニューラルネットが木より良くなる**（差 −0.025、−0.017。標準誤差0.006〜0.007）。
- つまり、世界Aでは、**どのデータ量でも、MLP（予報・季節だけ）が最良かそれに近い**。「深層学習は大量のデータが必要」という通説は、**この問題の小さなネットには当てはまりません**。むしろ、**少ないデータでは木のほうが過学習しやすい**。

**世界B（予報が完璧）**:
- **1.5年**: **GRU 0.276 が最良**。HGB（全特徴量）0.293、直線回帰 0.300、MLP 0.309。GRUとHGBの差は −0.017（標準誤差 0.007）。
- **3年**: HGB（全特徴量）**0.241** が最良。GRU は 0.249（+0.008、標準誤差 0.004）。
- **5〜10年**: GRU 0.226／0.212 と、HGB 0.228／0.216 は**ほぼ同じ**（差 −0.003、−0.004。標準誤差 0.005、0.002）。
- **GRUは、データが少なくても崩れません**。手作りの特徴量を作る手間なしで、1.5年でも HGB に並びます。ただし、**3年以上では、HGB（手作りの特徴量）が同等以上**。「少量データでは、特徴量を作り込むほうが有利」という通説も、ここでは**1.5年では逆**でした。
- **上限2（0.193）との差**: 10年で GRU は +0.019。1.5年だと +0.083。**データが増えると、隠れた状態をより正確に読める**。

In [ ]:
piv = e2.pivot_table(index="手法", columns="発電量のノイズ", values="MAE").loc[["HGB（予報・季節だけ）", "HGB（全特徴量）", "GRU（生の窓）", "上限2"]].round(3)
gap = (piv.loc["HGB（予報・季節だけ）"] - piv.loc["HGB（全特徴量）"]).round(3)
display(piv)
print("隠れた状態を読む価値（HGB予報・季節だけ − HGB全特徴量）:")
display(gap.to_frame("MAEの差").T)

**出力の見方**（世界B・学習10年・発電量のノイズを1%〜10%に変える）:
- ノイズが大きいほど、**どの手法も上限2に近づき、手法の差が縮みます**。
- **隠れた状態を読む価値**（HGB予報・季節だけ − HGB全特徴量）: ノイズ1%で **0.099**、3%で 0.058、6%で 0.031、10%で **0.018**。**ノイズが大きいと、隠れた状態を読んでも、ノイズに埋もれて**、得が小さくなります。
- **GRU と HGB（全特徴量）**は、どのノイズでもほぼ同じ（差 −0.009〜+0.003）。ノイズ1%のときだけ GRU がわずかに良い（−0.009、標準誤差 0.000）。
- 現実の発電量のノイズは、日射予報・パネル特性・機器の揺らぎなど、**ここで想定した3%よりずっと大きいかもしれません**。その場合、**隠れた状態を読む価値は、さらに小さい**と考えるべきです。

## 試してみよう（演習）
1. **もっと短く**: `run_phase4.e1(years=(0.75, 1.5), seeds=[1_000_001], n_seeds=1)` を実行して、半年・1.5年の違いを見てください（実行に数分かかります）。分かるのは、データが少なすぎると、どの手法も直線回帰を超えられなくなる境目です。
2. **ネットを小さく**: GRUの `hidden=8` で、1.5年の世界Bの成績を比べてください。分かるのは、データが少ないときの、ネットの大きさの影響です。
3. **ノイズと予報**: `make_series(..., gen_noise=0.1)` の普通の世界Aで、E1と同じ比較をして、手法の差がどうなるか予想してから確かめてください。

## まとめ
- 普通の世界では、**MLP（予報・季節だけ）が、どのデータ量でも最良かそれに近い**。1.5年では、木（全特徴量）は直線回帰より悪く、**過学習**する。
- 予報が完璧な世界では、**GRUは1.5年でも崩れず、HGBに並ぶか上回る**。3年以上では、手作り特徴量のHGBが同等以上。
- 「深層学習は大量のデータが必要」は、**この問題の小さなネットでは当てはまらない**。むしろ、**少ないデータでは、特徴量の多い木が過学習する**。
- **ノイズが大きいほど、隠れた状態を読む価値は小さい**（1%で0.099、10%で0.018）。
- **実データ並みの短さ（1.5年）では、複雑な手法は基準（直線回帰）に勝てないことが多い**。実データでの期待を、ここで調整しておく。

次の Phase5 では、**確率予測**（分位点回帰）を扱います。点の予測だけでなく、「どのくらいの範囲に入るか」を、予測区間で表します。